In [3]:
!pip install gradio openai

In [ ]:
import os
import gradio as gr
from openai import OpenAI


#API Setup 

os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

BASE_URL = "https://api.mistral.ai/v1"
MODEL_NAME = "mistral-small-2603"


client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)


#System Prompt & Domain Restriction
SYSTEM_PROMPT = """
You are a dedicated Sports & Athletics Assistant.
Your domain is strictly limited to sports, athletic events, players, rules, and games.

RULES:
1. Answer ONLY questions related to sports and physical games.
2. If the user asks anything outside of sports (e.g., coding, math, general science, non-sports history), POLITELY DECLINE and state that you can only answer sports-related questions.
3. Keep responses concise, clear, and accurate.
"""

#Chat Logic Function
def predict(message, history, temperature, max_tokens, top_p):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    #Reconstructing past conversation history
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": message})

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            stream=True
        )

        partial_message = ""
        for chunk in response:
            if chunk.choices[0].delta.content is not None:
                partial_message += chunk.choices[0].delta.content
                yield partial_message

    except Exception as e:
        yield f"⚠️ Error: {str(e)}"

#Gradio UI Interface
demo = gr.ChatInterface(
    fn=predict,
    title="🏀 TinyLlama Sports Chatbot",
    description="Domain-Specific AI Chatbot powered by TinyLlama (1.1B).",
    textbox=gr.Textbox(placeholder="Ask a sports question..."),
    additional_inputs=[
        gr.Slider(minimum=0.1, maximum=1.0, value=0.7, step=0.05, label="Temperature"),
        gr.Slider(minimum=64, maximum=1024, value=256, step=64, label="Max New Tokens"),
        gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.05, label="Top-P"),
    ],
    examples=[
        ["Who holds the record for most centuries in international cricket?"],
        ["Explain the offside rule in football."],
        ["How do I solve a quadratic equation?"]  # Test out-of-domain rejection
    ]
)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5bb211b154923ac3f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
